# BoltzGen input cases

In [1]:
import time

import torch

from evedesign.models.boltzgen import BoltzGenGenerator
from evedesign.system import (
    AtomBond, Interaction, Ligand, Protein, SecondaryStructure, System,
)
from evedesign.structure import Structure, StructureFile

assert torch.cuda.is_available(), "boltzgen has no CPU path"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

NVIDIA L40S (8, 9)


In [2]:
NUM_DESIGNS = 50
BUDGET = 10
RESULTS = []
DESIGNS = {}


def run_case(name, system, protocol="protein-anything", fixed_pos=None,
             entities=(1,)):
    """entities defaults to the binder: every case is [target, binder],
    and the target is held fixed."""
    t0 = time.perf_counter()
    row = {"case": name, "designs": 0, "seq": "", "score": None, "error": ""}
    try:
        gen = BoltzGenGenerator(
            protocol=protocol,
            device="cuda",
            num_devices=2,
            budget=BUDGET,
            skip_inverse_folding=True,
            keep_tmp_dir=True,
        ).build(system)
        designs = gen.generate(num_designs=NUM_DESIGNS, fixed_pos=fixed_pos,
                               entities=entities)
        DESIGNS[name] = designs
        row["designs"] = len(designs)
        if designs:
            binder = designs[0][-1]
            if binder.rep is not None:
                row["seq"] = "".join(binder.rep)[:24]
            row["score"] = designs[0].score
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"[:140]
    row["min"] = round((time.perf_counter() - t0) / 60, 1)
    RESULTS.append(row)
    print(row)
    return row

In [3]:
TARGET_CIF = "target_cache/1g13.cif"

import biotite.structure as struc

sf = StructureFile(TARGET_CIF, format="cif")
chain_a = sf.get_model().get_chain("A")
aa = chain_a.atom_array[struc.filter_amino_acids(chain_a.atom_array)]

TARGET_SEQ = "".join(Structure(aa).res_df().res_name_oneletter)
TARGET_STRUCT = {"x": Structure(aa)}


def target(**kwargs):
    """1G13 chain A as a structural target, matching boltzgen's examples,
    which pass targets as file: entries rather than bare sequences."""
    return Protein(rep=TARGET_SEQ, id="target", structures=TARGET_STRUCT, **kwargs)


print(len(TARGET_SEQ), TARGET_SEQ[:40])


162 SSFSWDNCDEGKDPAVIRSLTLEPDPIIVPGNVTLSVMGS


## 1. Baseline: sequence target, variable-length binder

In [4]:
run_case("baseline", System([
    target(),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


2026-08-05 14:51:49.320 | INFO     | evedesign.models.boltz.convert_design:system_to_boltzgen_yaml:600 - System has 2 entities: 1 designed, 1 context
2026-08-05 14:51:49.324 | INFO     | evedesign.models.boltzgen:generate:307 - BoltzGen YAML written to /tmp/boltzgen_162sbxve/design_spec.yaml
2026-08-05 14:51:49.325 | INFO     | evedesign.models.boltzgen:generate:319 - Running BoltzGen: boltzgen run /tmp/boltzgen_162sbxve/design_spec.yaml --output /tmp/boltzgen_162sbxve/output --protocol protein-anything --num_designs 50 --num_workers 1 --inverse_fold_num_sequences 1 --budget 10 --design_checkpoints huggingface:boltzgen/boltzgen-1:boltzgen1_diverse.ckpt huggingface:boltzgen/boltzgen-1:boltzgen1_adherence.ckpt --inverse_fold_checkpoint huggingface:boltzgen/boltzgen-1:boltzgen1_ifold.ckpt --folding_checkpoint huggingface:boltzgen/boltzgen-1:boltz2_conf_final.ckpt --devices 2 --skip_inverse_folding
2026-08-05 15:05:19.434 | INFO     | evedesign.models.boltz.convert_design:_parse_diverse_se

{'case': 'baseline', 'designs': 10, 'seq': 'MTEDQLKSLIKQAEDFLKSQGISS', 'score': 0.59416, 'error': '', 'min': 13.5}


{'case': 'baseline',
 'designs': 10,
 'seq': 'MTEDQLKSLIKQAEDFLKSQGISS',
 'score': 0.59416,
 'error': '',
 'min': 13.5}

## 2. Motif scaffolding via fixed_pos

In [ ]:
MOTIF = "GGGGWKQLADQLYRAG"

run_case("motif", System([
    target(),
    Protein(rep=MOTIF, min_length=len(MOTIF), max_length=len(MOTIF),
            id="binder"),
]), fixed_pos={1: [5, 6, 7, 8, 9]})


2026-08-05 15:05:19.690 | INFO     | evedesign.models.boltz.convert_design:system_to_boltzgen_yaml:600 - System has 2 entities: 1 designed, 1 context
2026-08-05 15:05:19.695 | INFO     | evedesign.models.boltzgen:generate:307 - BoltzGen YAML written to /tmp/boltzgen_s8f425np/design_spec.yaml
2026-08-05 15:05:19.695 | INFO     | evedesign.models.boltzgen:generate:319 - Running BoltzGen: boltzgen run /tmp/boltzgen_s8f425np/design_spec.yaml --output /tmp/boltzgen_s8f425np/output --protocol protein-anything --num_designs 50 --num_workers 1 --inverse_fold_num_sequences 1 --budget 10 --design_checkpoints huggingface:boltzgen/boltzgen-1:boltzgen1_diverse.ckpt huggingface:boltzgen/boltzgen-1:boltzgen1_adherence.ckpt --inverse_fold_checkpoint huggingface:boltzgen/boltzgen-1:boltzgen1_ifold.ckpt --folding_checkpoint huggingface:boltzgen/boltzgen-1:boltz2_conf_final.ckpt --devices 2 --skip_inverse_folding


## 3. Secondary structure on the designed chain

In [ ]:
run_case("secondary_structure", System([
    target(),
    Protein(rep=None, min_length=60, max_length=60, id="binder",
            secondary_structure=[
                SecondaryStructure(pos=p, type="H") for p in range(10, 31)
            ]),
]))


## 4. Binding site on the target

In [ ]:
run_case("binding_types", System([
    target(interactions=[Interaction(id="site", pos=list(range(40, 61)))]),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


## 5. Cyclic binder

In [ ]:
run_case("cyclic", System([
    target(),
    Protein(rep=None, min_length=12, max_length=12, id="binder", cyclic=True),
]), protocol="peptide-anything")


## 6. Homo-oligomer target

In [ ]:
run_case("homo_oligomer", System([
    Protein(rep=TARGET_SEQ, id="target", copies=2),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))

## 7. Ligand target

In [ ]:
run_case("ligand", System([
    Ligand(rep="ATP", ligand_rep_type="ccd", id="lig"),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]), protocol="protein-small_molecule")

## 8. Sequence-only target

In [ ]:
run_case("sequence_target", System([
    Protein(rep=TARGET_SEQ, id="target"),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
print(df.to_string(index=False))
print()
print(f"{(df['designs'] > 0).sum()}/{len(df)} cases produced designs")
